# 02 · Preparação e Qualidade dos Dados

Objetivo: entender a estrutura da base, checar a qualidade dos dados, criar variáveis de apoio para a análise e salvar uma versão tratada em `data/processed/`.

Princípio adotado: **não alterar nem descartar dados sem justificativa de negócio**. Inconsistências são sinalizadas com *flags* e documentadas, para que a decisão fique transparente.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.nps_utils import (
    PROCESSED_DATA_PATH, ID_COLUMNS, TARGET_COLUMN, POST_EXPERIENCE_FEATURES,
    CUSTOMER_FEATURES, ORDER_FEATURES, LOGISTICS_FEATURES, SERVICE_FEATURES,
    load_raw_data, classify_nps,
)

pd.set_option("display.max_columns", 50)
raw_df = load_raw_data()
print(f"Linhas: {raw_df.shape[0]} | Colunas: {raw_df.shape[1]}")
raw_df.head()

Linhas: 2500 | Colunas: 19


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


## 1. Estrutura e tipos

In [2]:
# Tipos de dados e organização das colunas por "momento da jornada"
column_groups = {
    "Identificadores": ID_COLUMNS,
    "Cliente": CUSTOMER_FEATURES,
    "Pedido": ORDER_FEATURES,
    "Logística": LOGISTICS_FEATURES,
    "Atendimento": SERVICE_FEATURES,
    "Pós-experiência (não usar para prever)": POST_EXPERIENCE_FEATURES,
    "Target": [TARGET_COLUMN],
}
pd.DataFrame(
    [(group, col, str(raw_df[col].dtype)) for group, cols in column_groups.items() for col in cols],
    columns=["grupo", "coluna", "tipo"],
)

,grupo,coluna,tipo
0,Identificadores,customer_id,int64
1,Identificadores,order_id,int64
2,Cliente,customer_age,int64
3,Cliente,customer_region,object
4,Cliente,customer_tenure_months,int64
5,Pedido,order_value,float64
6,Pedido,items_quantity,int64
7,Pedido,discount_value,float64
8,Pedido,payment_installments,int64
9,Logística,delivery_time_days,int64


## 2. Completude e unicidade

In [3]:
missing_values = raw_df.isna().sum().sum()
duplicated_rows = raw_df.duplicated().sum()
duplicated_orders = raw_df["order_id"].duplicated().sum()
duplicated_customers = raw_df["customer_id"].duplicated().sum()

print(f"Valores ausentes: {missing_values}")
print(f"Linhas duplicadas: {duplicated_rows}")
print(f"order_id repetidos: {duplicated_orders}")
print(f"customer_id repetidos: {duplicated_customers}  -> cada cliente aparece com um único pedido")

Valores ausentes: 0
Linhas duplicadas: 0
order_id repetidos: 0
customer_id repetidos: 0  -> cada cliente aparece com um único pedido


A base está **completa** (sem nulos) e **sem duplicidades**. Cada linha é um pedido de um cliente distinto — portanto não há histórico por cliente, e cada linha pode ser tratada como uma observação independente.

## 3. Faixas de valores (estatística descritiva)

In [4]:
raw_df.drop(columns=ID_COLUMNS).describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
customer_age,2500.0,43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2500.0,61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_value,2500.0,434.26,289.77,7.76,220.24,375.52,577.29,1983.81
items_quantity,2500.0,3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,2500.0,29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,2500.0,6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2500.0,8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2500.0,2.19,1.45,0.00,1.00,2.00,3.00,8.00
freight_value,2500.0,38.22,12.08,2.62,29.93,38.50,46.27,76.13
delivery_attempts,2500.0,2.01,0.82,1.00,1.00,2.00,3.00,3.00


In [5]:
raw_df["customer_region"].value_counts()

customer_region
Sul             521
Sudeste         520
Norte           506
Nordeste        485
Centro-Oeste    468
Name: count, dtype: int64

Os valores estão dentro de faixas plausíveis: idade entre 18 e 69 anos, nota NPS entre 0 e 10, nenhum valor negativo. As cinco regiões têm volumes parecidos (~20% cada), o que facilita comparações.

## 4. Regras de consistência de negócio

Além de faixas, checamos se as variáveis "conversam" entre si:

In [6]:
# Cada regra descreve algo que, pela lógica do negócio, não deveria acontecer
consistency_rules = {
    "atraso maior que o tempo total de entrega": raw_df["delivery_delay_days"] > raw_df["delivery_time_days"],
    "desconto maior que o valor do pedido": raw_df["discount_value"] > raw_df["order_value"],
    "tempo de resolução > 0 sem nenhum contato com o SAC": (raw_df["resolution_time_days"] > 0) & (raw_df["customer_service_contacts"] == 0),
    "reclamações registradas sem nenhum contato com o SAC": (raw_df["complaints_count"] > 0) & (raw_df["customer_service_contacts"] == 0),
    "nota NPS fora de 0-10": ~raw_df["nps_score"].between(0, 10),
}
pd.DataFrame(
    {
        "linhas": {rule: int(mask.sum()) for rule, mask in consistency_rules.items()},
        "% da base": {rule: round(100 * mask.mean(), 1) for rule, mask in consistency_rules.items()},
    }
)

,linhas,% da base
atraso maior que o tempo total de entrega,121,4.8
desconto maior que o valor do pedido,35,1.4
tempo de resolução > 0 sem nenhum contato com o SAC,511,20.4
reclamações registradas sem nenhum contato com o SAC,531,21.2
nota NPS fora de 0-10,0,0.0


**Interpretação e decisão:**

| Achado | Possível explicação | Decisão |
|---|---|---|
| Atraso > tempo de entrega (~5%) | `delivery_time_days` pode representar o prazo *prometido* e não o realizado, ou o atraso pode ser contado a partir de outro marco | Manter; criar flag `flag_atraso_inconsistente`; recomendar validar a definição com a área de logística |
| Desconto > valor do pedido (~1%) | `order_value` pode ser o valor líquido (já com desconto) | Manter; criar flag; validar definição com o financeiro |
| Resolução/reclamações sem contato no SAC (~20%) | Reclamações podem chegar por outros canais (marketplace, redes sociais, Reclame Aqui) e serem resolvidas sem contato direto | Manter — é plausível e informativo |

Nenhuma linha foi removida: as inconsistências são pequenas, não afetam a target e removê-las poderia enviesar a análise. O ponto vira **recomendação de governança de dados** (dicionário com definições precisas de cada campo).

## 5. Valores extremos (outliers)

In [7]:
# Regra do IQR (1,5 x intervalo interquartil) para contar valores extremos por variável
# (repeat_purchase_30d é binária 0/1, então não faz sentido falar em outlier nela)
numeric_columns = raw_df.drop(columns=ID_COLUMNS + ["customer_region", "repeat_purchase_30d"]).columns
outlier_counts = {}
for column in numeric_columns:
    q1, q3 = raw_df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[column] = int(((raw_df[column] < q1 - 1.5 * iqr) | (raw_df[column] > q3 + 1.5 * iqr)).sum())

pd.Series(outlier_counts, name="qtd_outliers").sort_values(ascending=False).to_frame()

,qtd_outliers
customer_service_contacts,176
discount_value,125
order_value,84
complaints_count,29
delivery_delay_days,17
freight_value,12
customer_age,0
customer_tenure_months,0
items_quantity,0
payment_installments,0


Os valores extremos aparecem principalmente em `customer_service_contacts`, `discount_value` e `order_value`, e em menor número em `complaints_count`, `delivery_delay_days` e `freight_value`. São **eventos reais e relevantes** (pedidos grandes, atrasos longos, clientes com muitas reclamações) — justamente os casos que mais interessam ao negócio. **Decisão: manter.** Os modelos usados no notebook 04 (regressão linear e árvores) lidam bem com essa faixa de valores.

## 6. Engenharia de variáveis de apoio

Criamos variáveis que traduzem os números em **conceitos de negócio**, facilitando a EDA e a comunicação:

In [8]:
df = raw_df.copy()

# --- Target em categorias (Detrator / Neutro / Promotor) ---
df["nps_category"] = classify_nps(df["nps_score"])
df["is_detractor"] = (df["nps_category"] == "Detrator").astype(int)

# --- Logística ---
df["is_on_time"] = (df["delivery_delay_days"] == 0).astype(int)  # entregue no prazo?
df["delay_band"] = pd.cut(
    df["delivery_delay_days"], bins=[-1, 0, 1, 2, 3, 100],
    labels=["No prazo", "1 dia", "2 dias", "3 dias", "4+ dias"],
)

# --- Atendimento ---
df["complaints_band"] = pd.cut(
    df["complaints_count"], bins=[-1, 1, 2, 3, 4, 5, 100],
    labels=["0-1", "2", "3", "4", "5", "6+"],
)
df["contacts_band"] = pd.cut(
    df["customer_service_contacts"], bins=[-1, 0, 1, 2, 3, 100],
    labels=["0", "1", "2", "3", "4+"],
)
df["resolution_band"] = pd.cut(
    df["resolution_time_days"], bins=[-1, 2, 5, 8, 100],
    labels=["Até 2 dias", "3-5 dias", "6-8 dias", "9+ dias"],
)

# --- Perfil do cliente ---
df["age_band"] = pd.cut(df["customer_age"], bins=[17, 25, 35, 45, 55, 100],
                        labels=["18-25", "26-35", "36-45", "46-55", "56+"])
df["tenure_band"] = pd.cut(df["customer_tenure_months"], bins=[0, 12, 24, 60, 1000],
                           labels=["Até 1 ano", "1-2 anos", "2-5 anos", "5+ anos"])
df["order_value_band"] = pd.cut(df["order_value"], bins=[0, 150, 300, 500, 800, 10_000],
                                labels=["Até R$150", "R$150-300", "R$300-500", "R$500-800", "R$800+"])

# --- Flags de qualidade de dados (seção 4) ---
df["flag_atraso_inconsistente"] = (df["delivery_delay_days"] > df["delivery_time_days"]).astype(int)
df["flag_desconto_inconsistente"] = (df["discount_value"] > df["order_value"]).astype(int)

df[["nps_score", "nps_category", "delay_band", "complaints_band", "contacts_band", "age_band", "tenure_band"]].head()

,nps_score,nps_category,delay_band,complaints_band,contacts_band,age_band,tenure_band
0,6.9,Detrator,2 dias,3,0,56+,1-2 anos
1,2.4,Detrator,4+ dias,3,0,18-25,Até 1 ano
2,4.8,Detrator,1 dia,6+,4+,46-55,5+ anos
3,5.9,Detrator,2 dias,4,1,46-55,5+ anos
4,6.1,Detrator,1 dia,3,1,56+,2-5 anos


In [9]:
# Salva a base tratada para os próximos notebooks
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Base tratada salva em: {PROCESSED_DATA_PATH.relative_to(PROCESSED_DATA_PATH.parents[2])}")
print(f"Formato final: {df.shape}")

Base tratada salva em: data/processed/nps_tratado.csv
Formato final: (2500, 31)


## Resumo da preparação

- Base de **2.500 pedidos**, 19 colunas originais, **sem nulos e sem duplicidades**;
- Inconsistências pontuais de definição (atraso × tempo de entrega, desconto × valor) foram **sinalizadas, não removidas**;
- Outliers mantidos por representarem eventos reais de negócio;
- Criadas faixas (atraso, reclamações, contatos, idade, tempo de casa, valor) e a categoria NPS para a EDA;
- `repeat_purchase_30d` e `csat_internal_score` estão marcadas como **pós-experiência** e não entrarão no modelo preditivo.